In [11]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)

# =========================
# 🔥 DEVICE
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🔥 Using device: {device}")

# =========================
# 🔹 MODEL (FIXED FP ISSUE)
# =========================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ FIX: NO FP16 HERE
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

# ✅ IMPORTANT
tokenizer.pad_token = tokenizer.eos_token

# =========================
# 📂 LOAD DATASET
# =========================
dataset = load_dataset("json", data_files="making v1 dataset/v1_dataset.json")

print(dataset)  # 🔍 Debug structure

# =========================
# 🔧 FORMAT FUNCTION
# =========================
def format_data(example):
    response_text = " ".join(example["output"])

    prompt = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{response_text}
"""
    return {"text": prompt}


# =========================
# 🔧 TOKENIZE FUNCTION
# =========================
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()  # ✅ CRITICAL

    return tokens


# =========================
# 🔁 HANDLE DATASET TYPE
# =========================
if "train" in dataset:
    dataset = dataset["train"]

# Apply formatting
dataset = dataset.map(format_data)

# Apply tokenization
tokenized_dataset = dataset.map(tokenize, batched=True)

# =========================
# ⚙️ TRAINING CONFIG
# =========================
training_args = TrainingArguments(
    output_dir="./v1-model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=1e-5,
    logging_steps=10,
    save_steps=100,
    fp16=False,                # ✅ FIXED
    max_grad_norm=1.0,
    report_to="none"
)

# =========================
# 🏋️ TRAINER
# =========================
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args
)



🔥 Using device: cuda


DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 2514
    })
})


Map: 100%|██████████| 2514/2514 [00:00<00:00, 5994.81 examples/s]


RuntimeError: You can't move a model that has some modules offloaded to cpu or disk.

In [10]:
# =========================
# 🚀 TRAIN
# =========================
trainer.train()



  0%|          | 0/942 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 12.00 GiB of which 0 bytes is free. Of the allocated memory 18.66 GiB is allocated by PyTorch, and 15.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# =========================
# 💾 SAVE MODEL
# =========================
trainer.save_model("./v1-final-model")
tokenizer.save_pretrained("./v1-final-model")

print("✅ Training Complete!")